# Language Drift Direction Ablation — SE.LLMA Weight-Space Programming

**Hypothesis**: Targeted direction-ablation edits reduce Swedish language drift in
Qwen2.5-0.5B-Instruct, but improvements reflect failure removal rather than new capability.

**Hardware target**: Google Colab Free (T4 GPU, ~15 GB VRAM, fp16).

**Estimated runtime**: ~30–60 minutes.

## Instructions
1. Mount Google Drive (cell below) to persist results across session disconnects.
2. Run cells in order. Each cell checkpoints results to Drive.
3. If the session disconnects after cell N, re-run cells 1–N (they are idempotent).
4. Final results are in `results/summary.json` and `results/figures/`.

In [ ]:
# CELL 1: Mount Google Drive and set working directory
import os

USE_DRIVE = True  # Set False to skip Drive and use /content directly

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/sellma_experiment'
else:
    WORK_DIR = '/content/sellma_experiment'

os.makedirs(WORK_DIR, exist_ok=True)
print(f'Working directory: {WORK_DIR}')

In [ ]:
# CELL 2: Clone repo and install dependencies
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/sellma_experiment.git'  # TODO: replace with your repo URL
REPO_DIR = os.path.join(WORK_DIR, 'repo')

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
print(f'Repo at: {REPO_DIR}')

# Install pinned dependencies
os.system('pip install -q -r requirements.txt')
print('Dependencies installed.')

In [ ]:
# CELL 3: Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. This will be slow.')

In [ ]:
# CELL 4: Run unit tests
import subprocess
result = subprocess.run(['python', '-m', 'pytest', 'tests/', '-v', '--tb=short'], capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError('Unit tests failed. Fix before proceeding.')

In [ ]:
# CELL 5: Build data splits (saves to configs/data_split.json)
import sys, json
sys.path.insert(0, '.')

split_path = 'configs/data_split.json'
if os.path.exists(split_path):
    print('Splits already built, loading...')
    with open(split_path) as f:
        splits = json.load(f)
else:
    from src.data_utils import build_splits
    splits = build_splits(dev_size=50, test_size=100, seed=42)

print(f"dev_sv: {len(splits['dev_sv'])}, test_sv: {len(splits['test_sv'])}")
print(f"Leakage check passed (enforced in build_splits).")

In [ ]:
# CELL 6: Load model (checkpoint: saves revision to configs/model_config.json)
import torch
from src.model_utils import load_model_and_tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
model, tokenizer = load_model_and_tokenizer(device=device, dtype=dtype)
print('Model loaded.')

In [ ]:
# CELL 7: Compute language direction (FEEDBACK-BASED — declared)
import json
from src.data_utils import load_splits, build_qa_prompts
from src.edit_utils import compute_language_direction, layer_direction_scores

splits = load_splits()
dev_sv = build_qa_prompts(splits['dev_sv'], lang='sv')
dev_en = build_qa_prompts(splits['dev_en'], lang='en')

sv_texts = [p['prompt'] for p in dev_sv]
en_texts = [p['prompt'] for p in dev_en]

print('NOTE: This step runs the model on dev prompts (feedback-based). Declared as such in PLAN.md.')
directions = compute_language_direction(model, tokenizer, sv_texts, en_texts, device=device)
scores = layer_direction_scores(directions, model)
print(f'Top-5 layers: {scores[:5]}')

# Save direction scores
import os
os.makedirs('configs', exist_ok=True)
with open('configs/direction_scores.json', 'w') as f:
    json.dump({'scores': scores[:10]}, f, indent=2)
print('Direction scores saved to configs/direction_scores.json')

In [ ]:
# CELL 8: Tune alpha on dev set
from src.edit_utils import apply_direction_ablation
from src.eval_pipeline import run_condition, evaluate_records

top_layer = scores[0][0]
target_layers = [top_layer]

tune_dir = 'results/dev_tune'
os.makedirs(tune_dir, exist_ok=True)

best_alpha, best_acc = 1.0, -1.0
for alpha in [0.5, 1.0, 2.0]:
    edited = apply_direction_ablation(model, directions, target_layers=target_layers,
                                       alpha=alpha, n_columns=64)
    recs = run_condition(edited, tokenizer, dev_sv[:20], f'dev_alpha_{alpha}',
                          max_new_tokens=60, device=device,
                          out_dir=__import__('pathlib').Path(tune_dir))
    m = evaluate_records(recs, 'sv')
    print(f'alpha={alpha}: dev lang-ID={m["lang_id_accuracy"]:.2%}')
    if m['lang_id_accuracy'] > best_acc:
        best_acc = m['lang_id_accuracy']
        best_alpha = alpha

print(f'\nBest alpha: {best_alpha} (dev lang-ID={best_acc:.2%})')

edit_cfg = {'target_layers': target_layers, 'alpha': best_alpha, 'n_cols': 64, 'seed': 42}
with open('configs/edit_config.json', 'w') as f:
    json.dump(edit_cfg, f, indent=2)

In [ ]:
# CELL 9: Run all conditions on TEST SET (touched once)
# This cell is the main evaluation. Run it once only.

import copy
import numpy as np
from src.edit_utils import (apply_direction_ablation, apply_random_direction_edit,
                              apply_sign_flipped_edit, count_edited_params)
from src.metrics import stratify_by_base_response, compute_perplexity
from src.lora_utils import train_lora_baseline

test_sv = build_qa_prompts(splits['test_sv'], lang='sv')
test_en = build_qa_prompts(splits['test_en'], lang='en')
ppl_texts = [s['text'] for s in splits['perplexity_sv']]

test_out = __import__('pathlib').Path('results/test_sv')
test_out.mkdir(parents=True, exist_ok=True)
test_en_out = __import__('pathlib').Path('results/test_en')
test_en_out.mkdir(parents=True, exist_ok=True)

summary = {}

# C0: baseline
c0 = run_condition(model, tokenizer, test_sv, 'C0_baseline',
                   max_new_tokens=100, device=device, out_dir=test_out)
summary['C0_baseline'] = evaluate_records(c0, 'sv')
print(f"C0: {summary['C0_baseline']['lang_id_accuracy']:.2%}")

# C1
c1 = run_condition(model, tokenizer, test_sv, 'C1_sysprompt',
                   system_prompt='Svara alltid på svenska.',
                   max_new_tokens=100, device=device, out_dir=test_out)
summary['C1_sysprompt'] = evaluate_records(c1, 'sv')
print(f"C1: {summary['C1_sysprompt']['lang_id_accuracy']:.2%}")

# C2
c2 = run_condition(model, tokenizer, test_sv, 'C2_strong_sysprompt',
                   system_prompt='Du är en svensk assistent. Svara alltid på svenska.',
                   max_new_tokens=100, device=device, out_dir=test_out)
summary['C2_strong_sysprompt'] = evaluate_records(c2, 'sv')
print(f"C2: {summary['C2_strong_sysprompt']['lang_id_accuracy']:.2%}")

# C3: targeted edit
c3_model = apply_direction_ablation(model, directions, target_layers=target_layers,
                                     alpha=best_alpha, n_columns=64)
edit_info = count_edited_params(model, c3_model)
print(f'C3 edit: {edit_info["changed_params"]:,} params ({edit_info["fraction_changed"]:.4%})')
c3 = run_condition(c3_model, tokenizer, test_sv, 'C3_direction_ablation',
                   max_new_tokens=100, device=device, out_dir=test_out)
summary['C3_direction_ablation'] = evaluate_records(c3, 'sv')
strata = stratify_by_base_response(c0, c3, expected_lang='sv')
summary['C3_direction_ablation']['stratum_A'] = strata.get('A', {})
summary['C3_direction_ablation']['stratum_B'] = strata.get('B', {})
print(f"C3: {summary['C3_direction_ablation']['lang_id_accuracy']:.2%}")
print(f"  Stratum A (n={strata.get('A',{}).get('n',0)}): edit_acc={strata.get('A',{}).get('edit_accuracy',0):.2%}")
print(f"  Stratum B (n={strata.get('B',{}).get('n',0)}): edit_acc={strata.get('B',{}).get('edit_accuracy',0):.2%}")

# C4: random controls (5 seeds)
c4_accs = []
for seed_i in range(5):
    rand_m = apply_random_direction_edit(model, directions[target_layers[0]],
                                          target_layers=target_layers,
                                          alpha=best_alpha, n_columns=64, seed=seed_i)
    c4_recs = run_condition(rand_m, tokenizer, test_sv, f'C4_random_s{seed_i}',
                             max_new_tokens=100, device=device, out_dir=test_out)
    c4_accs.append(evaluate_records(c4_recs, 'sv')['lang_id_accuracy'])
summary['C4_random_control'] = {
    'lang_id_accuracy': float(np.mean(c4_accs)),
    'lang_id_std': float(np.std(c4_accs)),
    'n_seeds': 5,
}
print(f"C4 random: {summary['C4_random_control']['lang_id_accuracy']:.2%} ± {summary['C4_random_control']['lang_id_std']:.2%}")

# C5: sign-flipped
c5_model = apply_sign_flipped_edit(model, directions, target_layers=target_layers, alpha=best_alpha, n_columns=64)
c5 = run_condition(c5_model, tokenizer, test_sv, 'C5_sign_flipped',
                   max_new_tokens=100, device=device, out_dir=test_out)
summary['C5_sign_flipped'] = evaluate_records(c5, 'sv')
print(f"C5 sign-flip: {summary['C5_sign_flipped']['lang_id_accuracy']:.2%}")

# C6: LoRA
try:
    c6_model = train_lora_baseline(copy.deepcopy(model), tokenizer, dev_sv,
                                   num_epochs=3, lr=2e-4, r=8, seed=42, device=device)
    c6 = run_condition(c6_model, tokenizer, test_sv, 'C6_lora',
                       max_new_tokens=100, device=device, out_dir=test_out)
    summary['C6_lora'] = evaluate_records(c6, 'sv')
    print(f"C6 LoRA: {summary['C6_lora']['lang_id_accuracy']:.2%}")
except Exception as e:
    print(f'C6 LoRA failed: {e}')

# English regression
for cond_name, m_model in [('C0_baseline', model), ('C3_direction_ablation', c3_model)]:
    en_recs = run_condition(m_model, tokenizer, test_en, f'{cond_name}_en',
                            max_new_tokens=100, device=device, out_dir=test_en_out)
    en_m = evaluate_records(en_recs, 'en')
    summary[cond_name]['english_lang_id'] = en_m['lang_id_accuracy']
    print(f'{cond_name} English lang-ID: {en_m["lang_id_accuracy"]:.2%}')

# Perplexity
for cond_name, m_model in [('C0_baseline', model), ('C3_direction_ablation', c3_model)]:
    ppl = compute_perplexity(m_model, tokenizer, ppl_texts[:50], device=device)
    summary[cond_name]['perplexity_sv'] = ppl
    print(f'{cond_name} PPL_sv: {ppl:.2f}')

# Save summary (checkpoint)
import json
with open('results/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSummary saved to results/summary.json')

In [ ]:
# CELL 10: Analysis and figures
import subprocess
result = subprocess.run(['python', '-I', 'scripts/analyze_results.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# CELL 11: Display figures inline
from IPython.display import Image, display
import os

for fig in ['results/figures/lang_id_comparison.png', 'results/figures/stratification.png']:
    if os.path.exists(fig):
        display(Image(fig))
    else:
        print(f'Figure not found: {fig}')

In [ ]:
# CELL 12: Print summary table
import json
with open('results/summary.json') as f:
    summary = json.load(f)

print(f'{"Condition":<30} {"LangID%":>8} {"CI_lo":>8} {"CI_hi":>8} {"PPL_sv":>8} {"EN%":>8}')
print('-' * 80)
for cond, m in summary.items():
    if 'lang_id_accuracy' not in m:
        continue
    pct = m['lang_id_accuracy'] * 100
    lo = m.get('lang_id_ci_lower', float('nan')) * 100
    hi = m.get('lang_id_ci_upper', float('nan')) * 100
    ppl = m.get('perplexity_sv', float('nan'))
    en = m.get('english_lang_id', float('nan'))
    if en != float('nan'): en *= 100
    print(f'{cond:<30} {pct:>7.1f}% {lo:>7.1f}% {hi:>7.1f}% {ppl:>8.1f} {en:>7.1f}%')

# Stratification summary for C3
c3 = summary.get('C3_direction_ablation', {})
sa = c3.get('stratum_A', {})
sb = c3.get('stratum_B', {})
print('\nC3 Stratification (failure removal analysis):')
print(f'  Stratum A (base→Swedish, n={sa.get("n",0)}): edit lang-ID = {sa.get("edit_accuracy",0)*100:.1f}%')
print(f'  Stratum B (base→English, n={sb.get("n",0)}): edit lang-ID = {sb.get("edit_accuracy",0)*100:.1f}%')
print('\nIf only Stratum B improves: failure removal.')
print('If Stratum A also improves: evidence of new capability.')